In [ ]:
import os
from typing import Any, cast

import torch
import torch.nn as nn
from torchvision.models import resnet18
import genesis as gs
import numpy as np

from collections import defaultdict

from tensordict.nn import TensorDictModule, TensorDictSequential
from tensordict.nn.distributions import NormalParamExtractor

from tensordict import TensorDict
from torchrl.data import Composite, Bounded

from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (
    Compose,
    DoubleToFloat,
    ObservationNorm,
    StepCounter,
    TransformedEnv,
    Resize,
)
from torchrl.envs import EnvBase
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator, IndependentNormal
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm


d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\modules\mcts\scores.py:574: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  PUCT = functools.partial(PUCTScore, c=5)  # AlphaGo default value
d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\modules\mcts\scores.py:575: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB = functools.partial(UCBScore, c=math.sqrt(2))  # default from Auer et al. 2002
d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\modules\mcts\scores.py:576: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB1_TUNED = functools.partial(
d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\modules\mcts\scores.py:579: FutureWarning: functools.partia

In [2]:
gs.init(backend=gs.cpu, logging_level="warning")

In [ ]:
# gs.destroy()

[Genesis] [20:56:13] [INFO] 💤 Exiting Genesis and caching compiled kernels...


In [3]:
YOUBOT_DESCRIPTION = r"D:\ros\src\youbot_description"

JOINTS = [
    "wheel_joint_fl",
    "wheel_joint_fr",
    "wheel_joint_bl",
    "wheel_joint_br",
    "arm_joint_1",
    "arm_joint_2",
    "arm_joint_3",
    "arm_joint_4",
    "arm_joint_5",
    "gripper_finger_joint_l",
    "gripper_finger_joint_r",
]

joint2id = {k: i for i, k in enumerate(JOINTS)}

LINKS = [
    "arm_link_5",
    "plate_link",
    "gripper_finger_link_l",
    "gripper_finger_link_r",
]

link2id = {k: i for i, k in enumerate(LINKS)}

BATCH_RENDERER = False

num_joints = len(JOINTS)

device = torch.device("cuda:0")

In [4]:
class VecEnv(EnvBase):
    def __init__(self, n_envs: int, device=None):
        super().__init__(batch_size=torch.Size([n_envs]), device=device)
        self.set_device = device
        self.n_envs = n_envs

        self._set_seed(42)
        
        self.scene = gs.Scene(
            show_viewer=False,
            viewer_options=gs.options.ViewerOptions(max_FPS=120),
            vis_options=gs.options.VisOptions(env_separate_rigid=True),
        )
        self.plane = self.scene.add_entity(gs.morphs.Plane())

        self.bot = self.scene.add_entity(
            gs.morphs.URDF(
                file=os.path.join(YOUBOT_DESCRIPTION, "robots/youbot.urdf"),
                pos=(0, 0, 0.1),
                merge_fixed_links=False
            ),
        )

        self.cylinder_height = torch.tensor([0.1], device=self.set_device)
        self.plate_height = 0.025

        self.target_cylinder = self.scene.add_entity(
            gs.morphs.Cylinder(pos=(1.0, 1.0, 0.05), height=self.cylinder_height, radius=0.015),
        )

        self.reward_flags = TensorDict({
            "has_opened": torch.zeros(self.n_envs, dtype=torch.bool, device=self.set_device),
            "has_lifted": torch.zeros(self.n_envs, dtype=torch.bool, device=self.set_device),
        }, batch_size=[self.n_envs], device=self.set_device)

        self.bot_dofs_idx = [self.bot.get_joint(name).dofs_idx_local[0] for name in JOINTS]
        self.bot_links_idx = [self.bot.get_link(name).idx_local for name in LINKS]

        if BATCH_RENDERER:
            renderer_opts = gs.sensors.BatchRendererCameraOptions(
                res=(640, 480),
                pos=(0.05, 0.0, 0.05),  # Offset from link frame
                lookat=(0.25, -0.02, 0.6),  # Look direction
                up=(0.0, 0.0, 1.0),
                entity_idx=self.bot.idx,  # Attach to robot
                link_idx_local=self.bot_links_idx[link2id["arm_link_5"]],  # End-effector link
                lights=[
                    {
                        "pos": (2.0, 2.0, 5.0),
                        "color": (1.0, 1.0, 1.0),
                        "intensity": 1.0,
                        "directional": True,
                        "castshadow": True,
                    }
                ],
            )
        else:
            renderer_opts = gs.sensors.RasterizerCameraOptions(
                res=(640, 480),
                pos=(0.05, 0.0, 0.05),  # Offset from link frame
                lookat=(0.25, -0.02, 0.6),  # Look direction
                up=(0.0, 0.0, 1.0),
                entity_idx=self.bot.idx,  # Attach to robot
                link_idx_local=self.bot_links_idx[link2id["arm_link_5"]],  # End-effector link
                # use_rasterizer=True,
            )

        self.camera = self.scene.add_sensor(cast(Any, renderer_opts))

        self.scene.build(n_envs=self.n_envs, env_spacing=(1.0, 1.0))

        self.set_initial_state()

        self.observation_spec = Composite(
            observation=Bounded(
                low=0, high=255, 
                shape=(self.n_envs, 3, 480, 640), 
                dtype=torch.uint8,
                device=self.set_device
            ),
            shape=(self.n_envs,),
            device=self.set_device
        )

        self.action_spec = Bounded(
            low=-float("inf"), high=float("inf"), 
            shape=(self.n_envs, num_joints),
            dtype=torch.float32,
            device=self.set_device
        )

        self.reward_spec = Bounded(
            low=-float("inf"), high=float("inf"),
            shape=(self.n_envs, 1),
            device=self.set_device
        )
        
        self.done_spec = Composite(
            done=Bounded(
                low=0, high=1,
                shape=(self.n_envs, 1),
                dtype=torch.bool,
                device=self.set_device
                ),
            terminated=Bounded(
                low=0, high=1,
                shape=(self.n_envs, 1),
                dtype=torch.bool,
                device=self.set_device
                ),
            shape=(self.n_envs,),
            device=self.set_device
        )

    def set_initial_state(self, envs_idx=None):
        if envs_idx is None:
            envs_idx = list(range(self.n_envs))

        pos = torch.stack([
            torch.empty(2, device=self.set_device).uniform_(-0.5, 0.5, generator=self.rngs[i])
            for i in range(self.n_envs)
        ])

        pos[:, 0] += 1.0

        pos = torch.cat([pos, (self.cylinder_height / 2).expand(pos.shape[0]).unsqueeze(-1)], dim=-1)
        
        self.target_cylinder.set_pos(pos=pos[envs_idx], envs_idx=envs_idx)

    def get_cylinder_slope(self):
        q = self.target_cylinder.get_quat().to(device=self.set_device)
        q = q / q.norm(dim=-1, keepdim=True)

        q_xyz = q[:, :3]
        q_w = q[:, -1].unsqueeze(-1)

        v0 = torch.tensor([1.0, 0.0, 0.0], device=self.set_device)
        v0 = v0.expand_as(q_xyz)

        t = 2 * torch.cross(q_xyz, v0, dim=-1)
        v = v0 + q_w * t + torch.cross(q_xyz, t, dim=-1)

        v = v / v.norm(dim=-1, keepdim=True)

        dot = (v * v0).sum(dim=-1).clamp(-1.0, 1.0)
        theta = torch.acos(dot)

        theta_deg = torch.rad2deg(theta)

        return theta_deg

    def get_gripper_state(self):
        bots_links_pos, _ = self.get_main_poses()
        l_finger_pose = bots_links_pos[:, link2id["gripper_finger_link_l"]]
        r_finger_pose = bots_links_pos[:, link2id["gripper_finger_link_r"]]
        l_anchor_pose = self.bot.get_joint("gripper_finger_joint_l").get_anchor_pos().to(self.set_device)
        r_anchor_pose = self.bot.get_joint("gripper_finger_joint_r").get_anchor_pos().to(self.set_device)

        l_distance = torch.sum((l_finger_pose - l_anchor_pose) ** 2, -1) ** 0.5
        r_distance = torch.sum((r_finger_pose - r_anchor_pose) ** 2, -1) ** 0.5

        alpha_closed = 0.001
        alpha_opened = 0.012

        return TensorDict({
            "l_closed": torch.round(l_distance, decimals=3) <= alpha_closed,
            "r_closed": torch.round(r_distance, decimals=3) <= alpha_closed,
            "l_opened": torch.round(l_distance, decimals=3) >= alpha_opened,
            "r_opened": torch.round(r_distance, decimals=3) >= alpha_opened,
        }, batch_size=bots_links_pos.shape[0], device=self.set_device)

    def get_main_poses(self):
        bots_links_pos = self.bot.get_links_pos(self.bot_links_idx)
        cylinder_pos = self.target_cylinder.get_pos()

        return bots_links_pos.to(self.set_device), cylinder_pos.to(self.set_device)

    def _get_observation(self):
        images = self.camera.read()
        return torch.tensor(images.rgb, dtype=torch.uint8, device=self.set_device).permute(0, 3, 1, 2)
    
    def _compute_reward(self):
        bots_links_pos, cylinder_pos = self.get_main_poses()
        hands_pos = bots_links_pos[:, link2id["arm_link_5"]]

        # отнимаем из награды дистанцию от руки до цилиндра
        distance = torch.sum((hands_pos - cylinder_pos) ** 2, -1, keepdim=True) ** 0.5
        total_reward = -distance        

        # даём награду за первое открытие хватателя
        gripper_state = self.get_gripper_state()

        has_opened = gripper_state["l_opened"] & gripper_state["r_opened"] & (~self.reward_flags["has_opened"])
        if torch.any(has_opened):
            total_reward[has_opened] += 5.0
            self.reward_flags["has_opened"][has_opened] = True

        # отнимаем из награды чрезмерный наклон цилиндра 
        cylinder_slope = self.get_cylinder_slope()

        alpha_slope = 15
        beta_slope = 0,2
        tilted = cylinder_slope > alpha_slope

        if torch.any(tilted):
            total_reward[tilted] -= (cylinder_slope - alpha_slope) * beta_slope
        
        # даём награду за первое поднятие цилиндра в воздух
        cylinder_pos[:, -1] -= self.cylinder_height / 2

        alpha_lifted = 0.05
        lifted = (cylinder_pos[:, -1] >= alpha_lifted) & (~self.reward_flags["has_lifted"])

        if torch.any(lifted):
            total_reward[lifted] += 5.0
            self.reward_flags["has_lifted"][lifted] = True

        # награждаем за приближение цилиндра в воздухе к платформе
        plate_pos = bots_links_pos[:, link2id["plate_link"]]
        plate_pos[:, -1] += self.plate_height

        in_air = cylinder_pos[:, -1] >= alpha_lifted

        if torch.any(in_air):
            distance = torch.sum((plate_pos - cylinder_pos) ** 2, -1, keepdim=True) ** 0.5
            total_reward[in_air] += 1.0 - distance[in_air]

        return total_reward
    
    def _is_terminated(self):
        bots_links_pos, cylinder_pos = self.get_main_poses()
        plate_pos = bots_links_pos[:, link2id["plate_link"]]

        plate_pos[:, -1] += self.plate_height
        cylinder_pos[:, -1] -= self.cylinder_height / 2
        cylinder_angle_error = self.get_cylinder_slope()

        horizontal_distance = torch.sum((plate_pos[:, :-1] - cylinder_pos[:, :-1]) ** 2, -1) ** 0.5
        vertical_distance = torch.abs(plate_pos[:, -1] - cylinder_pos[:, -1])

        horizontal_alpha = 0.12
        vertical_alpha = 0.005 
        angle_alpha = 2.0

        return ((horizontal_distance <= horizontal_alpha)
                & (vertical_distance <= vertical_alpha)
                & (cylinder_angle_error <= angle_alpha))

    def _reset(self, tensordict=None):
        obs = self._get_observation()
        done = torch.zeros((self.n_envs, 1), dtype=torch.bool, device=self.set_device)

        if tensordict is not None:
            done = tensordict["done"]
            self.scene.reset(envs_idx=done.nonzero().squeeze().cpu().numpy())
            self.set_initial_state(envs_idx=done.nonzero().squeeze().cpu().numpy())
        
        return TensorDict({
            "observation": obs,
            "done": done,
            "terminated": done
        }, batch_size=self.batch_size, device=self.set_device)
    
    def _step(self, tensordict):
        action = tensordict["action"]
        
        self.bot.control_dofs_velocity(
            velocity=action.squeeze().cpu().numpy(),
            dofs_idx_local=self.bot_dofs_idx,
        )
        self.scene.step()

        obs = self._get_observation()
        reward = self._compute_reward()
        done = self._is_terminated()

        out_tensordict = tensordict.clone()

        if torch.any(done):
            reset = self._reset(tensordict)

            out_tensordict["next"] = {
                "observation": reset["observation"],
                "reward": reward,
                "done": reset["done"],
                "terminated": reset["terminated"]
            }
        else:
            out_tensordict["next"] = {
                "observation": obs,
                "reward": reward,
                "done": done,
                "terminated": done
            }

        out_tensordict["reward"] = reward

        return out_tensordict

    def _set_seed(self, seed: int):
        self.np_random = np.random.default_rng(seed)
        self.rngs = []
        for i in range(self.n_envs):
            g = torch.Generator(device=self.set_device)
            g.manual_seed(seed + i)
            self.rngs.append(g)

In [ ]:
env = VecEnv(4, device=device)
transformed_env = TransformedEnv(
    env,
    Resize(320, 320, in_keys="observation", out_keys="observation", interpolation="bilinear")
)

[Genesis] [22:04:17] [WARNING] Falling back to legacy URDF parser. Default values of physics properties may be off:
Error: no decoder found for mesh file 'D:\ros/src/youbot_description/meshes/youbot_base/base_convex.dae' - Element name 'base_convex', id 0
[Genesis] [22:04:18] [WARNING] Link 'base_laser_front_link' has dubious inertia [ixx=1.000e-01,iyy=1.000e-01,izz=1.000e-01] compared to the estimate from geometry [ixx=8.786e-05,iyy=8.816e-05,izz=5.720e-05] given material density 1500.000.
[Genesis] [22:04:18] [WARNING] Link 'arm_link_5' has dubious center of mass [x=0.000, y=0.001, z=-0.017] compared to the bounding box from geometry [x=(-0.024, 0.030), y=(-0.048, 0.048), z=(-0.063, -0.022)].
[Genesis] [22:04:18] [WARNING] Link 'gripper_palm_link' has dubious inertia [ixx=1.000e-02,iyy=1.000e-02,izz=1.000e-02] compared to the estimate from geometry [ixx=8.177e-05,iyy=3.777e-05,izz=7.321e-05] given material density 1500.000.
[Genesis] [22:04:18] [WARNING] Link 'gripper_finger_link_l' 

In [20]:
frames_per_batch = 64
total_frames = 100_000

num_epochs = 5
sub_batch_size = 8

lr = 1e-4
gamma = 0.995
lmbda = 0.97

clip_epsilon = 0.1
entropy_eps = 1e-3

max_grad_norm = 0.5

action_dim = num_joints

In [17]:
class ResNetEncoder(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()

        self.backbone = resnet18(weights=None if not pretrained else "IMAGENET1K_V1", norm_layer=self.gn)

        self.backbone.maxpool = nn.Identity()

        self.features = nn.Sequential(*list(self.backbone.children())[:-1])
        # [B, 512, 1, 1]

        self.flatten = nn.Flatten()

    def gn(self, num_channels):
        return nn.GroupNorm(num_groups=32, num_channels=num_channels)

    def forward(self, x):
        # x: [B, 3, H, W]
        x = x / 255.0
        x = self.features(x)
        x = self.flatten(x)  # [B, 512]
        return x

In [21]:
encoder_net = ResNetEncoder().to(device=device)

actor_net = nn.Sequential(
    nn.Linear(512, 256),
    nn.Tanh(),
    nn.Linear(256, 256),
    nn.Tanh(),
    nn.Linear(256, 2 * action_dim),
).to(device=device)

critic_net = nn.Sequential(
    nn.Linear(512, 256),
    nn.Tanh(),
    nn.Linear(256, 256),
    nn.Tanh(),
    nn.Linear(256, 1),
).to(device=device)

feature_extractor = TensorDictModule(
    encoder_net, in_keys=["observation"], out_keys=["hidden"]
)

actor_head = TensorDictModule(
    actor_net, in_keys=["hidden"], out_keys=["loc_scale"]
)

extractor = TensorDictModule(
    NormalParamExtractor(), in_keys=["loc_scale"], out_keys=["loc", "scale"]
)

value_head = TensorDictModule(
    critic_net, in_keys=["hidden"], out_keys=["state_value"]
)

actor_module = TensorDictSequential(feature_extractor, actor_head, extractor)
value_module = TensorDictSequential(feature_extractor, value_head)

policy_module = ProbabilisticActor(
    module=actor_module,
    in_keys=["loc", "scale"],
    spec=transformed_env.action_spec,
    distribution_class=IndependentNormal,
    return_log_prob=True,
)


In [22]:
collector = SyncDataCollector(
    transformed_env,
    policy_module,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
    split_trajs=False,
    device=device
)

replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=frames_per_batch),
    sampler=SamplerWithoutReplacement(),
)

advantage_module = GAE(
    gamma=gamma, lmbda=lmbda, value_network=value_module, average_gae=True
)

loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=clip_epsilon,
    entropy_bonus=bool(entropy_eps),
    entropy_coeff=entropy_eps,
    critic_coeff=1.0,
    loss_critic_type="smooth_l1",
)

optim = torch.optim.Adam(loss_module.parameters(), lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim, total_frames // frames_per_batch, 0.0
)

d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\collectors\_base.py:1045: DeprecationWarning: SyncDataCollector has been deprecated and will be removed in v0.13. Please use Collector instead.
  warnings.warn(
d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\collectors\_single.py:911: UserWarning: total_frames (100000) is not exactly divisible by frames_per_batch (64). This means 32 additional frames will be collected.To silence this message, set the environment variable RL_WARNINGS to False.
  warnings.warn(


In [ ]:
logs = defaultdict(list)
pbar = tqdm(total=total_frames)

for i, tensordict_data in enumerate(collector):

    tensordict_data = tensordict_data.reshape(-1, *tensordict_data.shape[2:])

    for _ in range(num_epochs):
        with torch.no_grad():
            advantage_module(tensordict_data.squeeze(0))

        data_view = tensordict_data.reshape(-1)
        replay_buffer.extend(data_view.cpu())

        # === PPO inner loop ===
        for _ in range(frames_per_batch // sub_batch_size):
            subdata = replay_buffer.sample(sub_batch_size)

            loss_vals = loss_module(subdata.to(device))

            loss = (
                loss_vals["loss_objective"]
                + loss_vals["loss_critic"]
                + loss_vals["loss_entropy"]
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_grad_norm)

            optim.step()
            optim.zero_grad()

    # === logging ===
    logs["reward"].append(tensordict_data["next", "reward"].mean().item())
    pbar.update(tensordict_data.numel())

    # === evaluation ===
    if i % 10 == 0:
        with set_exploration_type(ExplorationType.DETERMINISTIC), torch.no_grad():
            eval_rollout = transformed_env.rollout(200, policy_module)
            logs["eval_reward"].append(eval_rollout["next", "reward"].mean().item())
            del eval_rollout

    scheduler.step()


In [24]:
torch.save({
    "actor": policy_module.state_dict(),
    "critic": value_module.state_dict(),
    "optimizer": optim.state_dict(),
}, "checkpoint.pt")